# Data dimensions and coverage

This notebook documents what data enter the project, their dimensions, and when each factor and strategy becomes available. It deliberately avoids performance analysis. Raw factor returns come from the Ken French data files stored in `data/raw/`; recession dates come from the stored NBER indicator. The two processed panels must be rebuilt with Stages 1 and 2 described in the project README before this notebook is run. Returns are stored as decimals and dates are month-end unless stated otherwise.

In [1]:
# 0. Libraries and project modules
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from config import FACTORS, FACTOR_MOMENTUM_PANEL, MANAGED_FACTORS
from data import load_factor_returns, series_coverage, validate_unique_dates

## 1. Source and processed datasets

In [2]:
daily, monthly = load_factor_returns()
managed = validate_unique_dates(pd.read_parquet(MANAGED_FACTORS), 'managed-factor panel')
factor_momentum = validate_unique_dates(pd.read_parquet(FACTOR_MOMENTUM_PANEL), 'factor-momentum panel')

datasets = {
    'Raw daily factors': daily,
    'Raw monthly factors': monthly,
    'Processed managed factors': managed,
    'Processed factor momentum': factor_momentum,
}
dimensions = pd.DataFrame([
    {
        'dataset': name, 'rows': len(data), 'columns': len(data.columns),
        'start': data['date'].min(), 'end': data['date'].max(),
        'duplicate_dates': int(data['date'].duplicated().sum()),
    }
    for name, data in datasets.items()
])
display(dimensions)

,dataset,rows,columns,start,end,duplicate_dates
0,Raw daily factors,26296,10,1926-07-01,2026-07-31,0
1,Raw monthly factors,1201,11,1926-07-31,2026-07-31,0
2,Processed managed factors,1201,143,1926-07-31,2026-07-31,0
3,Processed factor momentum,1201,49,1926-07-31,2026-07-31,0


## 2. Factor-level source coverage

In [3]:
source_coverage = []
for factor, label in FACTORS.items():
    daily_coverage = series_coverage(daily, factor)
    monthly_coverage = series_coverage(monthly, factor)
    source_coverage.append({
        'factor': label,
        'daily_start': daily_coverage['start'],
        'daily_end': daily_coverage['end'],
        'daily_observations': daily_coverage['observations'],
        'monthly_start': monthly_coverage['start'],
        'monthly_end': monthly_coverage['end'],
        'monthly_observations': monthly_coverage['observations'],
    })
source_coverage = pd.DataFrame(source_coverage)
display(source_coverage)

,factor,daily_start,daily_end,daily_observations,monthly_start,monthly_end,monthly_observations
0,Mkt,1926-07-01,2026-07-31,26296,1926-07-31,2026-07-31,1201
1,SMB,1926-07-01,2026-07-31,26296,1926-07-31,2026-07-31,1201
2,HML,1926-07-01,2026-07-31,26296,1926-07-31,2026-07-31,1201
3,Mom,1926-11-03,2026-07-31,26195,1927-01-31,2026-07-31,1195
4,RMW,1963-07-01,2026-07-31,15876,1963-07-31,2026-07-31,757
5,CMA,1963-07-01,2026-07-31,15876,1963-07-31,2026-07-31,757


## 3. Volatility-managed strategy coverage

Lagged realized variance becomes available after one prior variance observation. AR(1) and GARCH require their own 120-month initialization periods.

In [4]:
strategy_coverage = []
for factor, label in FACTORS.items():
    row = {'factor': label}
    for method, column in {
        'lagged_rv': f'managed_{factor}',
        'ar1': f'managed_ar1_{factor}',
        'garch': f'managed_garch_{factor}',
    }.items():
        row.update(series_coverage(managed, column, method))
    strategy_coverage.append(row)
display(pd.DataFrame(strategy_coverage))

,factor,lagged_rv_start,lagged_rv_end,lagged_rv_observations,ar1_start,ar1_end,ar1_observations,garch_start,garch_end,garch_observations
0,Mkt,1926-08-31,2026-07-31,1200,1936-07-31,2026-07-31,1081,1936-07-31,2026-07-31,1081
1,SMB,1926-08-31,2026-07-31,1200,1936-07-31,2026-07-31,1081,1936-07-31,2026-07-31,1081
2,HML,1926-08-31,2026-07-31,1200,1936-07-31,2026-07-31,1081,1936-07-31,2026-07-31,1081
3,Mom,1927-01-31,2026-07-31,1195,1936-11-30,2026-07-31,1077,1936-11-30,2026-07-31,1077
4,RMW,1963-08-31,2026-07-31,756,1973-07-31,2026-07-31,637,1973-07-31,2026-07-31,637
5,CMA,1963-08-31,2026-07-31,756,1973-07-31,2026-07-31,637,1973-07-31,2026-07-31,637


## 4. Factor-momentum strategy coverage

In [5]:
return_columns = [column for column in factor_momentum if column.startswith('factor_momentum_')]
momentum_coverage = pd.DataFrame([
    {'strategy': column, **series_coverage(factor_momentum, column, 'return')}
    for column in return_columns
])
display(momentum_coverage)

,strategy,return_start,return_end,return_observations
0,factor_momentum_original,1927-07-31,2026-07-31,1189
1,factor_momentum_original_scaled_lagged_rv,1927-08-31,2026-07-31,1188
2,factor_momentum_original_scaled_ar1,1937-07-31,2026-07-31,1069
3,factor_momentum_original_scaled_garch,1937-07-31,2026-07-31,1069
4,factor_momentum_lagged_rv,1927-08-31,2026-07-31,1188
5,factor_momentum_lagged_rv_scaled_lagged_rv,1927-09-30,2026-07-31,1187
6,factor_momentum_lagged_rv_scaled_ar1,1937-08-31,2026-07-31,1068
7,factor_momentum_lagged_rv_scaled_garch,1937-08-31,2026-07-31,1068
8,factor_momentum_ar1,1937-07-31,2026-07-31,1069
9,factor_momentum_ar1_scaled_lagged_rv,1937-08-31,2026-07-31,1068


## 5. Structural validation

The processed panels must have unique dates and finite values wherever observations are present.

In [6]:
validation = pd.DataFrame([
    {
        'dataset': name,
        'duplicate_dates': int(data['date'].duplicated().sum()),
        'missing_dates': int(data['date'].isna().sum()),
        'nonfinite_numeric_values': int(sum(
            (~np.isfinite(data[column].dropna())).sum()
            for column in data.select_dtypes(include='number')
        )),
    }
    for name, data in datasets.items()
])
display(validation)

,dataset,duplicate_dates,missing_dates,nonfinite_numeric_values
0,Raw daily factors,0,0,0
1,Raw monthly factors,0,0,0
2,Processed managed factors,0,0,0
3,Processed factor momentum,0,0,0
